# Investigate betsson snapshots

Reads the parquet files written by `testing/scrape_betsson.ipynb` (or `session.get_odds(...)` directly) and explores them.

Each file under `data/snapshots/betsson_<sport>.parquet` accumulates rows across scrapes. Every row's `scraped_at` UTC timestamp identifies which snapshot it came from.

In [ ]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent
SNAPSHOTS = ROOT / "data" / "snapshots"
print("snapshots dir:", SNAPSHOTS)

In [ ]:
# List available betsson snapshot files
files = sorted(SNAPSHOTS.glob("betsson_*.parquet"))
for f in files:
    print(f"{f.name:<40} {f.stat().st_size:>10,} bytes")

In [ ]:
# Load every snapshot file into a single DataFrame (sport column comes from the data)
df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True) if files else pd.DataFrame()
print("total rows:", len(df))
if not df.empty:
    print("\nrows per sport:")
    print(df["sport"].value_counts())
    print("\nscrapes per sport:")
    print(df.groupby("sport")["scraped_at"].nunique())
df.head()

In [ ]:
# Most-recent snapshot per sport: the latest events list as currently observed
if not df.empty:
    latest_ts = df.groupby("sport")["scraped_at"].max().reset_index()
    latest = df.merge(latest_ts, on=["sport", "scraped_at"])
    print(f"latest snapshot rows: {len(latest)}")
    display(latest_ts)
    latest[["scraped_at", "sport", "league", "home_team", "away_team",
            "market", "home_odds", "draw_odds", "away_odds"]].head(20)

In [ ]:
# Odds movement: pick the event with the most repeated observations and trace it over time
if not df.empty:
    counts = df.groupby("event_id").size().sort_values(ascending=False)
    if len(counts) and counts.iloc[0] > 1:
        target = counts.index[0]
        movement = (
            df[df["event_id"] == target]
            .sort_values("scraped_at")
            [["scraped_at", "sport", "home_team", "away_team", "clock", "period",
              "market", "home_odds", "draw_odds", "away_odds"]]
        )
        print(f"event_id={target}: {len(movement)} observations")
        display(movement)
    else:
        print("no event has been scraped more than once yet — re-run the scraper a few times to see odds movement")

In [ ]:
# Summary: events per (sport, league)
if not df.empty:
    by_league = (
        df.drop_duplicates(subset=["sport", "event_id"])
          .groupby(["sport", "league"]).size()
          .sort_values(ascending=False)
    )
    print("unique events per (sport, league):")
    by_league.head(30)